# 03 Demo (Product & Demo Lead sandbox)
## Your job in this notebook
Build and polish the Gradio demo and the product story. Your files: `src/solarcast/demo.py`, this notebook, `main.ipynb`
and `README.md`. Edit `demo.py`, then re-run the cells below (autoreload picks up your changes).

## Setup (run this first)
This cell makes the notebook work both in Google Colab and on a laptop. In Colab it clones (or updates) the
GitHub repo into `/content/SolarCast-Lite`; locally it finds the repo root. Then it puts `src/` on the Python path
and turns on autoreload, so edits to `src/solarcast/*.py` are picked up without restarting.
If the repo is private, add a Colab Secret named `GITHUB_TOKEN` (see `docs/COLAB_GUIDE.md`). The token is never printed.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/nashwu0117/SolarCast-Lite.git"
IN_COLAB = "google.colab" in sys.modules


def _git(args, cwd=None, secret=None, check=True):
    """Run git, hiding the token if it shows up in any output."""
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if secret:
        out = out.replace(secret, "***")
    if check and r.returncode != 0:
        raise RuntimeError("git " + args[0] + " failed:\n" + out)
    return r.returncode, out


if IN_COLAB:
    REPO = Path("/content/SolarCast-Lite")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None  # no secret: fine for a public repo
    auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
    if not REPO.exists():
        _git(["clone", auth_url, str(REPO)], secret=token)
        _git(["remote", "set-url", "origin", REPO_URL], cwd=REPO)  # keep the token out of .git/config
    elif not (REPO / ".git").exists():
        print("Using the uploaded copy in", REPO, "(not a git clone, so no pull).")
    else:
        _git(["remote", "set-url", "origin", auth_url], cwd=REPO, secret=token)
        code, out = _git(["pull", "--ff-only"], cwd=REPO, secret=token, check=False)
        _git(["remote", "set-url", "origin", REPO_URL], cwd=REPO)
        if code != 0:
            print("git pull did not fast-forward (local edits?). Continuing with the current files.\n" + out)
    os.chdir(REPO)
    if importlib.util.find_spec("gradio") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
else:
    REPO = Path.cwd().resolve()
    while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
        REPO = REPO.parent
    os.chdir(REPO)

# Editable installs do not take effect in Colab without a runtime restart, so use the path directly.
sys.path.insert(0, str(REPO / "src"))
print("Repo:", REPO, "| Colab:", IN_COLAB)

# Pick up edits to src/solarcast/*.py without restarting. Colab's IPython 7 autoreload imports the
# removed `imp` module on Python 3.12+, so fall back gracefully instead of failing the whole notebook.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as exc:
    print(f"autoreload unavailable ({type(exc).__name__}); after editing src/, use Runtime > Restart session.")

In [ ]:
from solarcast.pipeline import run_experiment

res_a = run_experiment("A")
res_b = run_experiment("B")

### One prediction, without the UI

In [ ]:
from solarcast.demo import predict_at

t = res_a.splits["test"].index[500]
info = predict_at(res_a, t)
{k: v for k, v in info.items() if k not in ("recent_power", "weather_now")}

### Launch the demo
In Colab this creates a public `share=True` link. *Historical replay on held-out test data. Prototype.*

In [ ]:
import os

from solarcast.demo import build_demo

demo = build_demo({"A": res_a, "B": res_b})
# Set SOLARCAST_LAUNCH_DEMO=0 to build the app without launching it (used for automated checks).
if os.environ.get("SOLARCAST_LAUNCH_DEMO", "1") != "0":
    demo.launch(share=IN_COLAB)  # Colab: public link; locally: inline